        # KULA: top recurring issues and recommendations
Period: October 26–November 1, 2026 (Monday–Sunday), inclusive. Run all cells to refresh. EXCLUDED_SUBMITTERS currently contains {''} and applies to category counts, ML topics and recommendation rankings. This excludes blank submitter names, not Ozy. Reviewed themes below require this exact filter; changing it requires a new review. Matching ignores case and surrounding spaces, but requires a full-name match. Set EXCLUDED_SUBMITTERS = set() to include everyone. After changing this parameter, rerun all cells; old outputs do not reflect the new filter.

Method: count observation rows by their recorded **Area of Improvement**, not inferred semantic issues. Positive feedback can also appear in this CSV, so these counts are not confirmed defect counts. Each nonblank observation counts once; repeated submissions are retained. Blank categories become Unspecified. Ties use alphabetical order. Source CSV stays unchanged. Category baseline uses standard library. ML section requires numpy and scikit-learn; no external services receive data.

In [1]:
import csv
from collections import Counter
from datetime import datetime, date, timedelta
from pathlib import Path

CSV_PATH = Path(r'C:\ai_dashboard\dataset_kula\Live Chat- KULA Testing - Testing.csv')
START_DATE = date(2026, 10, 26)
END_DATE = date(2026, 11, 1)
EXCLUDED_SUBMITTERS = {''}
def submitter_is_excluded(value):
    return value.strip().casefold() in {name.strip().casefold() for name in EXCLUDED_SUBMITTERS}
assert START_DATE <= END_DATE

def parse_date(value):
    try:
        return datetime.strptime(value.strip(), '%m/%d/%Y').date()
    except ValueError:
        return None

def rank_categories(records):
    counts = Counter(r['category'] for r in records)
    return sorted(counts.items(), key=lambda item: (-item[1], item[0]))

def in_window(day):
    return day is not None and START_DATE <= day <= END_DATE


## Load and validate
CSV repeats Reporting Week header. Position 0 contains dates; position 1 contains week labels. Record numbers identify logical CSV records, not physical lines because fields contain newlines. Invalid dates and blank observations are excluded and reported.

In [2]:
records = []
invalid_dates = []
blank_observations = 0
outside_window = 0
excluded_submitters = 0
all_dates = []
with CSV_PATH.open(encoding='utf-8-sig', newline='') as handle:
    reader = csv.reader(handle)
    headers = next(reader)
    assert headers[0].strip() == 'Reporting Week'
    assert headers[2].strip() == 'Submitted By'
    assert headers[7].startswith('Area of Improvement')
    assert headers[8].startswith('Issue / Observation')
    total_rows = 0
    for record_number, row in enumerate(reader, start=2):
        total_rows += 1
        if len(row) != len(headers):
            raise ValueError(f'Unexpected column count at record {record_number}')
        day = parse_date(row[0])
        if day is None:
            invalid_dates.append(record_number)
            continue
        all_dates.append(day)
        if not in_window(day):
            outside_window += 1
            continue
        if submitter_is_excluded(row[2]):
            excluded_submitters += 1
            continue
        observation = row[8].strip()
        if not observation:
            blank_observations += 1
            continue
        records.append(dict(record=record_number, date=day, category=row[7].strip() or 'Unspecified', observation=observation))
print(f'Window: {START_DATE} through {END_DATE}')
print(f'Source rows: {total_rows}; included: {len(records)}; outside window: {outside_window}')
print(f'Invalid/missing dates: {len(invalid_dates)}; blank observations within window: {blank_observations}')
if all_dates:
    print(f'Source date coverage: {min(all_dates)} through {max(all_dates)}')
if invalid_dates:
    print('Excluded record numbers:', invalid_dates)
print(f'Excluded submitters within window: {excluded_submitters}; filter: {EXCLUDED_SUBMITTERS}')
assert total_rows == len(records) + outside_window + len(invalid_dates) + blank_observations + excluded_submitters


Window: 2026-10-26 through 2026-11-01
Source rows: 310; included: 110; outside window: 200
Invalid/missing dates: 0; blank observations within window: 0
Source date coverage: 2026-10-19 through 2026-11-04
Excluded submitters within window: 0; filter: {''}


In [3]:
ranking = rank_categories(records)
top_three = ranking[:3]
days = [START_DATE + timedelta(days=i) for i in range((END_DATE - START_DATE).days + 1)]
daily = Counter((r['category'], r['date']) for r in records)
if not records:
    print('No nonblank observations in selected window. No top 3 available.')
else:
    print('TOP RECORDED CATEGORIES (not confirmed defects)')
    for position, (category, count) in enumerate(top_three, start=1):
        print(f'{position}. {category}: {count} ({count / len(records):.1%} of included observations)')
    if len(ranking) > 3 and ranking[2][1] == ranking[3][1]:
        print('Tie at third place; alphabetical order selects displayed categories.')
    print('\nDAILY BREAKDOWN')
    print('Category | ' + ' | '.join(d.strftime('%a %m/%d') for d in days) + ' | Total')
    for category, count in top_three:
        values = [daily[category, day] for day in days]
        print(category + ' | ' + ' | '.join(map(str, values)) + f' | {count}')
    print('All categories | ' + ' | '.join(str(sum(r['date'] == d for r in records)) for d in days) + f' | {len(records)}')
    for day in days:
        if not any(r['date'] == day for r in records):
            print(f'{day}: no included observations; zero is not evidence of no issues.')
assert sum(count for _, count in ranking) == len(records)
for category, count in ranking:
    assert sum(daily[category, day] for day in days) == count


TOP RECORDED CATEGORIES (not confirmed defects)
1. Knowledge Base / FAQ: 109 (99.1% of included observations)
2. Menu / Button Optimization: 1 (0.9% of included observations)

DAILY BREAKDOWN
Category | Mon 10/26 | Tue 10/27 | Wed 10/28 | Thu 10/29 | Fri 10/30 | Sat 10/31 | Sun 11/01 | Total
Knowledge Base / FAQ | 24 | 25 | 20 | 20 | 20 | 0 | 0 | 109
Menu / Button Optimization | 1 | 0 | 0 | 0 | 0 | 0 | 0 | 1
All categories | 25 | 25 | 20 | 20 | 20 | 0 | 0 | 110
2026-10-31: no included observations; zero is not evidence of no issues.
2026-11-01: no included observations; zero is not evidence of no issues.


## Evidence for interpretation
First three source observations per winning category, in CSV order. Examples are not an exhaustive semantic summary. Review these before treating a category as a specific recurring defect. Submitter names and UID columns are not displayed.

In [4]:
for category, count in top_three:
    print(f'\n{category} ({count} observations)')
    examples = [r for r in records if r['category'] == category][:3]
    for row in examples:
        print(f"Record {row['record']} | {row['date']} | {row['observation']}")



Knowledge Base / FAQ (109 observations)
Record 127 | 2026-10-26 | Menanyakan "Berapa lama proses pengiriman Produk Official Store?" jawaban kula tidak sesuai dengan FAQ di mana kula menjawab FAQ pengiriman idul fitri maret 2026
Record 129 | 2026-10-26 | Menanyakan 'dana yang saya terima tidak sesuai pada saat pengajuan pinjaman' kula menjawab langsung mengarahkan untuk ke cs manual
Record 130 | 2026-10-26 | Menanyakan "Bagaimana jika pesanan sudah dibatalkan namun pengembalian dana belum masuk ke "Saldo" di aplikasi Akulaku?" kula menjawab "mohon maf, pesanan ini tidak memenuhi syarat pengembalian dana dan pengembalian dana tidak dapat dilakuka saat ini. jika ada perrtanyaan silakan hubungi cs manual."

Menu / Button Optimization (1 observations)
Record 128 | 2026-10-26 | Menanyakan "saya mau meminta bukti POD / penerimaan produk produk official store" Kula menjawab dengan memberikan pilihan pesanan campuran dan setelah memilih produk yang ditanyakan kula menjawab sesuai informasi pad

In [5]:
assert parse_date('10/26/2026') == START_DATE
assert parse_date('11/01/2026') == END_DATE
assert parse_date('not a date') is None
assert parse_date('') is None
assert in_window(START_DATE) and in_window(END_DATE)
assert not in_window(START_DATE - timedelta(days=1))
assert not in_window(END_DATE + timedelta(days=1))
assert not in_window(None)
assert rank_categories([]) == []
assert rank_categories([{'category': 'B'}, {'category': 'A'}]) == [('A', 1), ('B', 1)]
print('Boundary, invalid-date, empty-data, tie-order and aggregation checks passed.')


Boundary, invalid-date, empty-data, tie-order and aggregation checks passed.


## ML simulation: top recurring themes
Local TF-IDF + NMF discovers lexical topics from observation text, without using existing categories. Appropriate interpretable baseline for small unlabeled Indonesian text, not a validated semantic classifier. Compare 2–6 topics across three seeds using cosine silhouette; select highest mean score. Reconstruction error is diagnostic, not a selection criterion. Rank up to three topics by row count, not model weight.
Normalized duplicate texts fit once; repeated source rows still count as mentions. Positive and mixed feedback remain included, so review examples before calling themes defects. Topic keywords are provisional labels. Bootstrap intervals hold assignments fixed and measure sampling variability, not accuracy. Requires numpy and scikit-learn in selected kernel. Run all cells above first.

In [6]:
import re
import warnings
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF
from sklearn.metrics import silhouette_score
from sklearn.exceptions import ConvergenceWarning

STOP_WORDS = 'yang dan di ke dari untuk dengan pada ini itu ada adalah juga sudah tidak bisa dalam akan nya lebih atau karena jadi oleh saat saya kami anda aku akulaku chatbot ai'.split()
def normalize_text(text):
    return re.sub(r'\s+', ' ', re.sub(r'[^\w\s]', ' ', text.casefold())).strip()

texts = [normalize_text(r['observation']) for r in records]
unique_texts = list(dict.fromkeys(texts))
ml_ready = False
candidates = []
if len(unique_texts) < 4:
    print('Insufficient distinct observations: need at least four for topic comparison.')
else:
    vectorizer = TfidfVectorizer(ngram_range=(1, 2), stop_words=STOP_WORDS, sublinear_tf=True, max_features=5000)
    try:
        X = vectorizer.fit_transform(unique_texts)
    except ValueError as exc:
        print('Cannot construct vocabulary:', exc)
    else:
        fit_mask = np.asarray(X.getnnz(axis=1)).ravel() > 0
        X_fit = X[fit_mask]
        for k in range(2, min(6, X_fit.shape[0] - 1, X_fit.shape[1]) + 1):
            runs = []
            for seed in (7, 42, 99):
                model = NMF(n_components=k, init='random', random_state=seed, max_iter=1000)
                with warnings.catch_warnings(record=True) as caught:
                    warnings.simplefilter('always', ConvergenceWarning)
                    weights = model.fit_transform(X_fit)
                labels = weights.argmax(axis=1)
                if any(issubclass(w.category, ConvergenceWarning) for w in caught):
                    continue
                if len(set(labels)) != k or np.any(weights.sum(axis=1) == 0):
                    continue
                score = silhouette_score(X_fit, labels, metric='cosine')
                runs.append((score, seed, model))
            if len(runs) == 3:
                mean_score = float(np.mean([run[0] for run in runs]))
                candidates.append((mean_score, k, runs))
                print(f'Topics={k}, mean silhouette={mean_score:.3f}, range={min(r[0] for r in runs):.3f}..{max(r[0] for r in runs):.3f}, mean reconstruction error={np.mean([r[2].reconstruction_err_ for r in runs]):.3f}')
        if candidates:
            mean_score, selected_k, runs = max(candidates, key=lambda result: (result[0], -result[1]))
            _, selected_seed, model = max(runs, key=lambda run: (run[0], -run[1]))
            ml_ready = True
            print(f'Selected {selected_k} topics, seed {selected_seed}. No ground-truth accuracy available.')
            if mean_score <= 0:
                print('WARNING: weak lexical separation; do not treat ranking as reliable issue taxonomy.')
        else:
            print('No valid converged topic comparison. No ML top 3 produced.')


Topics=2, mean silhouette=0.040, range=0.039..0.040, mean reconstruction error=9.813
Topics=3, mean silhouette=0.053, range=0.053..0.053, mean reconstruction error=9.638
Topics=4, mean silhouette=0.058, range=0.058..0.059, mean reconstruction error=9.496
Topics=5, mean silhouette=0.057, range=0.044..0.064, mean reconstruction error=9.396
Topics=6, mean silhouette=0.071, range=0.068..0.077, mean reconstruction error=9.287
Selected 6 topics, seed 42. No ground-truth accuracy available.


In [7]:
if ml_ready:
    all_X = vectorizer.transform(texts)
    all_weights = model.transform(all_X)
    assigned = all_weights.argmax(axis=1)
    assigned[all_weights.sum(axis=1) == 0] = -1
    counts = Counter(int(label) for label in assigned if label >= 0)
    ordered = sorted(counts.items(), key=lambda item: (-item[1], item[0]))
    terms = vectorizer.get_feature_names_out()
    rng = np.random.default_rng(42)
    bootstrap = rng.multinomial(len(records), [counts[t] / len(records) for t in range(selected_k)] + [float(np.mean(assigned == -1))], size=1000)
    print(f'Unassigned observations: {int(np.sum(assigned == -1))}')
    if len(ordered) > 3 and ordered[2][1] == ordered[3][1]:
        print('Tie at third place: lower topic ID displayed first; inspect other tied topics.')
    print('Full topic counts:', ordered)
    for rank, (topic, count) in enumerate(ordered[:3], start=1):
        keywords = terms[np.argsort(model.components_[topic])[::-1][:7]]
        members = np.flatnonzero(assigned == topic)
        interval = np.percentile(bootstrap[:, topic], [2.5, 97.5])
        print(f'\n{rank}. Topic {topic}: ' + ', '.join(keywords))
        print(f'{count} mentions ({count / len(records):.1%} of window rows); conditional bootstrap 95% count interval {interval[0]:.0f}–{interval[1]:.0f}')
        daily_counts = [sum(records[i]['date'] == day for i in members) for day in days]
        print(' | '.join(f'{day:%a %m/%d}: {value}' for day, value in zip(days, daily_counts)))
        assert sum(daily_counts) == count
        strength = all_weights[:, topic] / np.maximum(all_weights.sum(axis=1), 1e-12)
        representatives = sorted(members, key=lambda i: (-strength[i], i))
        seen = set()
        for i in representatives:
            if texts[i] in seen:
                continue
            seen.add(texts[i])
            row = records[i]
            print(f"Record {row['record']} | {row['date']} | {row['observation']}")
            if len(seen) == 3:
                break
    assert sum(counts.values()) + int(np.sum(assigned == -1)) == len(records)
    assert all(START_DATE <= row['date'] <= END_DATE for row in records)
    assert normalize_text(' TEST  text! ') == 'test text'
    print('\nML count reconciliation and date-window checks passed. Human issue-label review still required.')


Unassigned observations: 0
Full topic counts: [(1, 46), (2, 22), (5, 18), (0, 10), (4, 8), (3, 6)]

1. Topic 1: kula sesuai, jawaban kula, jawaban, sesuai faq, faq, sesuai, menanyakan
46 mentions (41.8% of window rows); conditional bootstrap 95% count interval 36–56
Mon 10/26: 6 | Tue 10/27: 8 | Wed 10/28: 10 | Thu 10/29: 10 | Fri 10/30: 12 | Sat 10/31: 0 | Sun 11/01: 0
Record 153 | 2026-10-27 | menanyakan 'Apa itu Lucky Draw?' jawaban kula sudah sesuai FAQ
Record 179 | 2026-10-28 | menanyakan 'min, bisa ga aku bayar pokoknya aja?' jawaban kula sudah sesuai FAQ
Record 205 | 2026-10-29 | Menanyakan "Apa yang dimaksud dengan Kredit Poin?" jawaban kula sudah sesuai FAQ

2. Topic 2: bahwa, pengguna, menjelaskan, dapat, mengarahkan, menjelaskan bahwa, tenor
22 mentions (20.0% of window rows); conditional bootstrap 95% count interval 15–30
Mon 10/26: 5 | Tue 10/27: 5 | Wed 10/28: 10 | Thu 10/29: 1 | Fri 10/30: 1 | Sat 10/31: 0 | Sun 11/01: 0
Record 133 | 2026-10-26 | Menanyakan cara mengubah

## Most-mentioned recommendations
Count the Suggestion / Recommendation column for the same inclusive October 26–November 1 (Monday–Sunday) window, independently of whether observation text is blank. One nonblank recommendation field equals one mention. Unicode, case and whitespace are normalized; punctuation is preserved. Paraphrases are not merged and multi-part recommendations are not split. This is exact-text frequency, not ML theme frequency. No-recommendation responses remain visible by default. Add reviewed exact phrases to EXCLUDE_RECOMMENDATIONS to exclude them; exclusion totals are reported. Display top three plus all ties at third place. Percentages use included nonblank recommendation rows.

In [8]:
import unicodedata
from collections import Counter

EXCLUDE_RECOMMENDATIONS = set()
def recommendation_key(text):
    return ' '.join(unicodedata.normalize('NFKC', text).casefold().split())

excluded_keys = {recommendation_key(text) for text in EXCLUDE_RECOMMENDATIONS}
recommendation_rows = []
recommendation_labels = {}
recommendation_audit = Counter()
with CSV_PATH.open(encoding='utf-8-sig', newline='') as handle:
    reader = csv.reader(handle)
    recommendation_headers = next(reader)
    assert recommendation_headers[2].strip() == 'Submitted By'
    recommendation_columns = [i for i, name in enumerate(recommendation_headers) if name.strip().startswith('Suggestion / Recommendation')]
    if len(recommendation_columns) != 1:
        raise ValueError('Expected exactly one Suggestion / Recommendation column')
    recommendation_column = recommendation_columns[0]
    for record_number, row in enumerate(reader, start=2):
        recommendation_audit['source_rows'] += 1
        if len(row) != len(recommendation_headers):
            raise ValueError(f'Unexpected column count at record {record_number}')
        day = parse_date(row[0])
        if day is None:
            recommendation_audit['invalid_dates'] += 1
            continue
        if not in_window(day):
            recommendation_audit['outside_window'] += 1
            continue
        if submitter_is_excluded(row[2]):
            recommendation_audit['excluded_submitters'] += 1
            continue
        text = row[recommendation_column].strip()
        key = recommendation_key(text)
        if not key:
            recommendation_audit['blank'] += 1
            continue
        if key in excluded_keys:
            recommendation_audit['excluded'] += 1
            continue
        recommendation_labels.setdefault(key, text)
        recommendation_rows.append((key, day, record_number))

recommendation_counts = Counter(key for key, _, _ in recommendation_rows)
recommendation_daily = Counter((key, day) for key, day, _ in recommendation_rows)
recommendation_ranking = sorted(recommendation_counts.items(), key=lambda item: (-item[1], item[0]))
recommendation_total = len(recommendation_rows)
print(f'Recommendation window: {START_DATE} through {END_DATE}')
print('Audit:', dict(recommendation_audit), '| included:', recommendation_total)
print(f'Distinct normalized recommendations: {len(recommendation_ranking)}')
recommendation_days = [START_DATE + timedelta(days=i) for i in range((END_DATE - START_DATE).days + 1)]
if not recommendation_ranking:
    print('No included recommendations in selected window.')
else:
    cutoff = recommendation_ranking[min(2, len(recommendation_ranking) - 1)][1]
    leaders = [(key, count) for key, count in recommendation_ranking if count >= cutoff]
    print('TOP RECOMMENDATIONS (including ties at cutoff)')
    for key, count in leaders:
        rank = 1 + sum(other_count > count for _, other_count in recommendation_ranking)
        print(f'\nRank {rank}: {recommendation_labels[key]}')
        print(f'{count} mentions ({count / recommendation_total:.1%} of included recommendations)')
        print(' | '.join(f'{day:%a %m/%d}: {recommendation_daily[key, day]}' for day in recommendation_days))
        print('Source record numbers:', [number for item, _, number in recommendation_rows if item == key])
    print('\nAll recommendation counts available in recommendation_ranking.')
assert sum(recommendation_counts.values()) == recommendation_total
assert recommendation_audit['source_rows'] == recommendation_total + sum(recommendation_audit[key] for key in ('invalid_dates', 'outside_window', 'blank', 'excluded', 'excluded_submitters'))
assert all(sum(recommendation_daily[key, day] for day in recommendation_days) == count for key, count in recommendation_ranking)
assert recommendation_key('  TEST\n recommendation ') == recommendation_key('test recommendation')
assert recommendation_key('') == ''
print('Recommendation count, daily-total and normalization checks passed.')


Recommendation window: 2026-10-26 through 2026-11-01
Audit: {'source_rows': 310, 'outside_window': 200, 'blank': 19} | included: 91
Distinct normalized recommendations: 22
TOP RECOMMENDATIONS (including ties at cutoff)

Rank 1: Tidak ada rekomendasi, sudah bagus
69 mentions (75.8% of included recommendations)
Mon 10/26: 14 | Tue 10/27: 14 | Wed 10/28: 13 | Thu 10/29: 15 | Fri 10/30: 13 | Sat 10/31: 0 | Sun 11/01: 0
Source record numbers: [137, 138, 139, 140, 141, 142, 143, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 158, 159, 160, 161, 162, 163, 164, 165, 166, 177, 178, 179, 180, 181, 182, 184, 185, 186, 192, 193, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 207, 208, 209, 210, 211, 217, 218, 219, 220, 221, 222, 223, 225, 226, 227, 228, 229, 230]

Rank 2: Seharusnya kula menjawab "Mohon maaf, untuk jenis permasalahan ini memerlukan pengecekan lebih lanjut. Silakan balas chat ini dengan ketik "Hubungi CS Livechat" untuk terhubung ke Customer Service atau h

        ## Reviewed improvement themes: 26 October–1 November 2026
Manual review assigns one primary theme to each actionable recommendation, not to every broad category or ML topic. These counts represent test observations, not unique incidents or production failure rates. Rows without recommendations and rows explicitly saying no recommendation are not counted as proven issues; their counts remain visible. Recommendations quoting FAQ responses require policy validation before implementation. This mapping is tied to the source fingerprint and current submitter filter below; changing source data, date range or filter requires a new review. Five Ozy-submitted rows with actionable recommendations are included; ten other Ozy rows have no recorded recommendation. No included records dated October 31 or November 1 were present when reviewed.

In [9]:
import hashlib

reviewed_hash = 'f35f7f6319d24d3590310af2c709e8ca2b3291a8fd3cca2de58d2cee263ac910'
assert hashlib.sha256(CSV_PATH.read_bytes()).hexdigest() == reviewed_hash, 'Source changed; repeat manual review.'
assert (START_DATE, END_DATE) == (date(2026, 10, 26), date(2026, 11, 1)), 'Reviewed themes require October 26 through November 1.'
assert {name.strip().casefold() for name in EXCLUDED_SUBMITTERS} == {''}, 'Reviewed themes require only blank submitter names excluded; repeat review if filter changes.'
reviewed_themes = [
    ('FAQ answer relevance and accuracy', {127, 129, 144, 183, 231, 233}, 'Retrieve relevant, approved FAQ guidance for shipping, loans, invoices, jobs and duplicate payments.'),
    ('Account and identity guidance', {133, 135}, 'Explain verified account-change steps and clarify account-freeze consequences without requesting sensitive data in chatbot.'),
    ('Billing discrepancy and collections guidance', {132, 136}, 'Avoid promising unverified SLIK changes or visit timing; check approved billing and collections policies.'),
    ('Official email channel guidance', {134}, 'Provide verified official email availability or explain alternatives before routing to Live Chat.'),
    ('Refund and cancellation guidance', {130, 131, 157, 176, 212, 213}, 'Give correct refund eligibility, status and balance guidance instead of unrelated refusal or order selection.'),
    ('Escalation for cases requiring other support', {215, 216, 224}, 'Route mismatched-item cases to human CS and BNC transfer cases to appropriate bank support.'),
    ('Official Store order selection', {128}, 'Restrict POD order options to relevant Official Store purchases.'),
    ('Delivered-item confirmation safety', {194}, 'Remind users to confirm receipt only after delivery and provide help for missing or problematic items.'),
]
reviewed_ids = [number for _, numbers, _ in reviewed_themes for number in numbers]
assert len(reviewed_ids) == len(set(reviewed_ids)) == 22
assert set(reviewed_ids) <= {row['record'] for row in records}
assert len(records) == 110 and outside_window == 200
with CSV_PATH.open(encoding='utf-8-sig', newline='') as handle:
    source_reader = csv.reader(handle)
    next(source_reader)
    source_recommendations = {number: row[9].strip() for number, row in enumerate(source_reader, 2)}
no_change = []
unrecorded = []
for row in records:
    number = row['record']
    recommendation = source_recommendations[number].casefold()
    if number in reviewed_ids:
        assert recommendation and not recommendation.startswith(('tidak ada rekomendasi', 'sudah sesuai faq')), f'Non-actionable theme: {number}'
    elif not recommendation:
        unrecorded.append(number)
    else:
        assert recommendation.startswith(('tidak ada rekomendasi', 'sudah sesuai faq')), f'Unreviewed recommendation: {number}'
        no_change.append(number)
assert len(no_change) == 69 and len(unrecorded) == 19
assert len(reviewed_ids) + len(no_change) + len(unrecorded) == len(records)
reviewed_days = [START_DATE + timedelta(days=i) for i in range((END_DATE - START_DATE).days + 1)]
record_dates = {row['record']: row['date'] for row in records}
print(f'{len(records)} observations: {len(reviewed_ids)} actionable, {len(no_change)} no change requested, {len(unrecorded)} without recommendation.')
for theme, numbers, action in sorted(reviewed_themes, key=lambda item: (-len(item[1]), item[0])):
    print(f'\n{theme}: {len(numbers)} ({len(numbers) / len(reviewed_ids):.1%} of actionable observations)')
    print('Daily:', ' | '.join(f'{day:%a %m/%d}: {sum(record_dates[number] == day for number in numbers)}' for day in reviewed_days))
    print('Source records:', sorted(numbers))
    print('Recommendation:', action)
assert sum(len(numbers) for _, numbers, _ in reviewed_themes) == len(reviewed_ids)
print('Reviewed classification, source fingerprint and filtering checks passed.')


110 observations: 22 actionable, 69 no change requested, 19 without recommendation.

FAQ answer relevance and accuracy: 6 (27.3% of actionable observations)
Daily: Mon 10/26: 3 | Tue 10/27: 0 | Wed 10/28: 1 | Thu 10/29: 0 | Fri 10/30: 2 | Sat 10/31: 0 | Sun 11/01: 0
Source records: [127, 129, 144, 183, 231, 233]
Recommendation: Retrieve relevant, approved FAQ guidance for shipping, loans, invoices, jobs and duplicate payments.

Refund and cancellation guidance: 6 (27.3% of actionable observations)
Daily: Mon 10/26: 2 | Tue 10/27: 2 | Wed 10/28: 0 | Thu 10/29: 2 | Fri 10/30: 0 | Sat 10/31: 0 | Sun 11/01: 0
Source records: [130, 131, 157, 176, 212, 213]
Recommendation: Give correct refund eligibility, status and balance guidance instead of unrelated refusal or order selection.

Escalation for cases requiring other support: 3 (13.6% of actionable observations)
Daily: Mon 10/26: 0 | Tue 10/27: 0 | Wed 10/28: 0 | Thu 10/29: 2 | Fri 10/30: 1 | Sat 10/31: 0 | Sun 11/01: 0
Source records: [215